# รวม v8.1 + pitty + Boom → v8.2

สถานะ: ตรวจข้อมูลจริงได้ แต่ยังไม่มี baseline v8.1 ในเครื่อง ไม่คัดลอกคะแนนจากรายงานและไม่รัน test ก่อนล็อก config

In [1]:
from pathlib import Path
import sys,json,subprocess,pandas as pd
from IPython.display import display,Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'modules/pipeline.py').is_file())
HERE=ROOT/'Boom/phase2.1/06_v82_integration'
sys.path.insert(0,str(HERE))
import preflight as pf
PY=ROOT/'Boom/venv-sign2say/bin/python'
display(pd.DataFrame([pf.readiness()]))

  version  ready  ... no_inference_run test_predictions_opened
0    v7.2  False  ...             True                   False

[1 rows x 7 columns]


## 0. ตรวจเวอร์ชันและข้อมูลรั่ว

ต้องมีโค้ดและ artifacts ของ v8.1 จริงก่อนเรียก tokenizer, calibration, grammar guard และ hand cue ผลการตรวจชื่อคนไม่ถือเป็นการยืนยันตัวบุคคล

In [2]:
RUN_PREFLIGHT=False
if RUN_PREFLIGHT:
    subprocess.run([str(PY),str(HERE/'preflight.py')],cwd=ROOT,check=True)
display(pd.read_csv(HERE/'outputs/eval_summary.csv').round(3))
display(pd.read_csv(HERE/'outputs/signer_audit.csv'))
display(Markdown((HERE/'outputs/training_file_duplicates.json').read_text()))

  person_folder  clips  duration_s  prior_development_duplicates
0          Boom      6      82.809                             6
1          Jack      5      49.858                             0
2         Pitty      7      45.147                             0
  person  ...                                             reason
0   Boom  ...  No user-verified mapping from folder person to...
1   Jack  ...  No user-verified mapping from folder person to...
2  Pitty  ...  No user-verified mapping from folder person to...

[3 rows x 4 columns]
<IPython.core.display.Markdown object>


**ตีความ:** hash ซ้ำยืนยันไฟล์เดียวกัน ส่วนไม่พบ hash ซ้ำไม่ได้ยืนยันว่า signer ไม่เคยอยู่ใน train; ไม่ใช้ใบหน้าเพื่อระบุตัวบุคคล

In [3]:
inventory=pd.read_csv(HERE/'outputs/eval_inventory.csv')
display(inventory[['person_folder','file','duration_s','fps','reference_filename','prior_development_duplicate','reference_needs_review','oov_status']].round(3))

   person_folder  ...                                         oov_status
0           Boom  ...  pending v8.1 tokenizer + 328-word variant mapping
1           Boom  ...  pending v8.1 tokenizer + 328-word variant mapping
2           Boom  ...  pending v8.1 tokenizer + 328-word variant mapping
3           Boom  ...  pending v8.1 tokenizer + 328-word variant mapping
4           Boom  ...  pending v8.1 tokenizer + 328-word variant mapping
5           Boom  ...  pending v8.1 tokenizer + 328-word variant mapping
6           Jack  ...  pending v8.1 tokenizer + 328-word variant mapping
7           Jack  ...  pending v8.1 tokenizer + 328-word variant mapping
8           Jack  ...  pending v8.1 tokenizer + 328-word variant mapping
9           Jack  ...  pending v8.1 tokenizer + 328-word variant mapping
10          Jack  ...  pending v8.1 tokenizer + 328-word variant mapping
11         Pitty  ...  pending v8.1 tokenizer + 328-word variant mapping
12         Pitty  ...  pending v8.1 tokenizer + 328

## 1. เฉลยและ OOV

ชื่อคลิปเป็นข้อความอ้างอิงตามผู้ใช้ ยังไม่เดาการแบ่ง gloss หรือ OOV จาก vocab v7.2 ต้องใช้ tokenizer และ variant mapping จริงของ v8.1 ชื่อที่ลงท้าย `_` แสดงเป็นรายการให้ตรวจโดยไม่แก้เฉลยเงียบ ๆ

In [4]:
plan=json.loads((HERE/'trial_plan.json').read_text())
display(pd.DataFrame.from_dict(plan['trials'],orient='index'))

   post.p_sign_min  ...  must_retest_combination_on_all_tune_sets
A             0.55  ...                                       NaN
B             0.60  ...                                       NaN
C              NaN  ...                                       NaN
D              NaN  ...                                       NaN
E              NaN  ...                                       NaN
F              NaN  ...                                       NaN
G              NaN  ...                                       NaN
H              NaN  ...                                      True

[8 rows x 13 columns]


## 2. แผน tune → freeze → test

A–G เปลี่ยนทีละตัวแปร ตรวจ C ว่าชน tau_accept ก่อน; H รวมเฉพาะค่าที่ผ่านและต้องทดสอบ H ซ้ำทุก tune set

เกณฑ์: ทุก tune set WER แย่ลงไม่เกิน 0.003, ค่าเฉลี่ยลดมากกว่า 0.002, accepted precision ≥0.90 ทุกชุด จากนั้นล็อก config และประเมิน test batch เดียวด้วย bootstrap 10,000 ครั้ง ส่วน Eval-Test ดูผลอย่างเดียว

ไม่มีการสร้าง config ชื่อ v8.2 ที่พร้อมใช้หรือ backup ชื่อ v8.1 จากไฟล์รุ่นเก่า

In [5]:
RUN_EXPERIMENT=False
if RUN_EXPERIMENT:
    pf.require_v81()
    raise RuntimeError('ต้องเชื่อม adapter เข้ากับ API การประเมินของ v8.1 จริงก่อน ไม่มี fallback ไป v7.2')

## ผลที่ยังไม่มี

ตาราง A–H, test CI, Eval-Test predictions และ config v8.2 ยังไม่ถูกสร้าง เพราะขาด v8.1 ไม่ใช้ตัวเลขจำลองแทน ผล preflight ด้านบนเป็นการอ่านไฟล์จริงเท่านั้น

## แผนทดลองฉบับล่าสุดและข้อจำกัดข้อมูล
เพิ่ม B2: `post.p_sign_min=0.70` และ E2: `post.min_frames=12` เป็นการทดลองแยกตัวแปร; H ต้องผ่านเกณฑ์บนทุกชุด tune อีกครั้งหลังรวม

Grammar weight 0.7 ยังปิดไว้จนตรวจแหล่งเรียนของ grammar ใน v8.1 จริงแล้ว ผลตรวจ grammar รุ่นเก่าใช้แทนไม่ได้

**ข้อจำกัด:** ตรวจพบ Boom ซ้ำกับคลิปพัฒนา 6/6 คลิป ส่วนการตรวจ exact-file ซ้ำกับวิดีโอฝึกตรวจได้ **0 ไฟล์** จาก manifest 14,143 แถว จึงห้ามสรุปว่าไม่รั่วไหลจากผล matches ว่าง

**ยังไม่มีผล v8.1/v8.2:** ต้องได้ source revision และ artifacts v8.1 จริงก่อน รัน baseline test และตัวเลือกที่ล็อกแล้วในขั้นสุดท้ายเท่านั้น
